# Question and why it matters
Does the chirality model predict edge-entry probability beyond coarse letter groups? A high pooled score need not imply within-group prediction. See [registration and recipe](README.md) and [scientific record](../../../notes/chirality-baselines-2026-10-15.md).

## Assumptions and definitions
Verified: the saved cohort, target values, calibration membership and chirality predictions reconstruct exactly (tolerance 1e-14). Fit group means on the same 300 calibration rules; score all 2,747 original held-out rules. Primary groups use rule[1]; secondary groups use rule[1:4], with training-only fallback. No grouping uses measured c. This is a reanalysis after inspection of the original plot.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import polars as pl
SEED = 20261046
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Cargo.toml").exists())
E = ROOT / "experiments/archive/2026-10-chirality-baselines/results"
paths = [E / "comparison.json", E / "groups.csv", E / "held_out_predictions.csv"]
assert all(p.is_file() for p in paths)
s = json.loads(paths[0].read_text())
assert s["seed"] == SEED
print("Registration date: 2026-10-15 NZDT; seed:", SEED)
print(json.dumps(s["provenance"], indent=2))
for p in paths:
    print(p.relative_to(ROOT), hashlib.sha256(p.read_bytes()).hexdigest())


Registration date: 2026-10-15 NZDT; seed: 20261046
{
  "git_commit": "b3a0f762ac2230b5df92188206a4d3a1081897be",
  "working_tree": "Uncommitted changes present; exact input/script hashes below",
  "command": "uv run experiments/2026-10-15-chirality-baselines/scripts/compare.py",
  "python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]",
  "numpy": "2.5.4",
  "polars": "2.0.0",
  "inputs_sha256": {
    "experiments/2026-10-13-probe-growth-census/results/c_model.csv": "5378ca19616dd6e4c5f894403cd74afa72e7d3092d18d600031b58150779b42e",
    "experiments/2026-10-13-probe-growth-census/results/probe_summary.csv": "c76c7927185b1cee273433c340cfa393fb16696fbe4a6f155fc49c33ef32cb5d",
    "experiments/2026-10-13-edge-statistics/results/edgestats_sample.csv": "976b1359224fdbf511ab1ac4d876854da0c9acf83e9eff5a39180e3845231464",
    "experiments/2026-10-13-probe-growth-census/scripts/c_model.py": "c754e2d7854ff9be115e9c89c1e1fbc8203ff1b9c42b40ca82e5b2ba040393cf",
    "uv.lock": "598ec0fc1978845

## Data and method
The producing script is `scripts/compare.py`; it writes per-rule predictions and subgroup statistics. Conventional R² uses the held-out mean only as a scoring denominator; strict R² compares with the training-mean predictor. RMSE and SSE compare actual predictions. Centered within-group R² subtracts each held-out group's mean from both observations and predictions, isolating discrimination from group-mean bias; these centering operations do not define a deployable predictor. Bootstrap intervals resample paired held-out rules within the two primary groups, keeping model fits fixed (4,000 draws).

## Results

In [2]:
table = pl.DataFrame([{"model": k, **{m: v[m] for m in ["n", "r2", "strict_r2", "sse", "rmse", "mae"]}} for k, v in s["overall"].items()])
print(table)
print(json.dumps(s["comparisons"], indent=2))
assert abs(s["overall"]["chirality"]["r2"] - 0.6600227968106389) < 1e-12


shape: (4, 7)
┌────────────┬──────┬───────────┬───────────┬───────────┬──────────┬──────────┐
│ model      ┆ n    ┆ r2        ┆ strict_r2 ┆ sse       ┆ rmse     ┆ mae      │
│ ---        ┆ ---  ┆ ---       ┆ ---       ┆ ---       ┆ ---      ┆ ---      │
│ str        ┆ i64  ┆ f64       ┆ f64       ┆ f64       ┆ f64      ┆ f64      │
╞════════════╪══════╪═══════════╪═══════════╪═══════════╪══════════╪══════════╡
│ constant   ┆ 2747 ┆ -0.004051 ┆ 0.0       ┆ 44.160921 ┆ 0.126791 ┆ 0.099676 │
│ letter1    ┆ 2747 ┆ 0.585589  ┆ 0.587261  ┆ 18.226949 ┆ 0.081457 ┆ 0.066907 │
│ letters123 ┆ 2747 ┆ 0.807648  ┆ 0.808424  ┆ 8.460185  ┆ 0.055496 ┆ 0.037915 │
│ chirality  ┆ 2747 ┆ 0.660023  ┆ 0.661395  ┆ 14.953126 ┆ 0.07378  ┆ 0.05574  │
└────────────┴──────┴───────────┴───────────┴───────────┴──────────┴──────────┘
{
  "letter1": {
    "sse_reduction": 0.17961444222175516,
    "sse_reduction_ci": [
      0.15916923770539063,
      0.20078881594334483
    ],
    "delta_r2": 0.07443428902422923,
    

Chirality improves on one-letter grouping but the three-letter lookup predicts more accurately. These are held-out predictions of c, not evidence for the origin of the growth exponent. The quoted R²=0.88 is not reproduced: existing results give R²=0.660 and correlation 0.869.

In [3]:
groups = pl.read_csv(E / "groups.csv")
print(groups.filter((pl.col("grouping") == "letter1") & (pl.col("model") != "constant")).select("group", "model", "n", "n_train_group", "r2", "rmse", "vs_baseline_sse_reduction"))
print(json.dumps(s["grouping"], indent=2))
print("Registered prediction outcomes:", s["registered_predictions"])


shape: (6, 7)
┌───────┬────────────┬──────┬───────────────┬───────────┬──────────┬───────────────────────────┐
│ group ┆ model      ┆ n    ┆ n_train_group ┆ r2        ┆ rmse     ┆ vs_baseline_sse_reduction │
│ ---   ┆ ---        ┆ ---  ┆ ---           ┆ ---       ┆ ---      ┆ ---                       │
│ str   ┆ str        ┆ i64  ┆ i64           ┆ f64       ┆ f64      ┆ f64                       │
╞═══════╪════════════╪══════╪═══════════════╪═══════════╪══════════╪═══════════════════════════╡
│ L     ┆ letter1    ┆ 1452 ┆ 173           ┆ -0.001194 ┆ 0.085419 ┆ 0.0                       │
│ L     ┆ letters123 ┆ 1452 ┆ 173           ┆ 0.369208  ┆ 0.067802 ┆ 0.369961                  │
│ L     ┆ chirality  ┆ 1452 ┆ 173           ┆ 0.18996   ┆ 0.076834 ┆ 0.190927                  │
│ R     ┆ letter1    ┆ 1295 ┆ 127           ┆ -0.000171 ┆ 0.076771 ┆ 0.0                       │
│ R     ┆ letters123 ┆ 1295 ┆ 127           ┆ 0.766059  ┆ 0.037129 ┆ 0.766099                  │
│ R     ┆ chiral

Chirality has nonzero skill inside the broad one-letter groups (R²=0.190 and 0.164), but centered within the three-letter groups its pooled R² is approximately zero. The latter describes residual discrimination; it does not establish that every individual group has exactly zero skill. The prediction that one-letter grouping lies within 0.05 of chirality fails; the prediction of less than 10% gain over three-letter grouping holds, with chirality actually worse.

## Caveats and controls
Synthetic controls test perfect, zero and adverse residual prediction, group-mean shifts, training-only fits, fallback and paired bootstrap reproducibility. They do not validate a physical mechanism. No trajectory estimator or fit window changed, so rotor/random-walk calibration was not rerun. The current cohort includes related/repeated rules; rule bootstrap assumes independent sampling units and conditions on this training sample. No between-seed uncertainty or fresh-rule confirmation is available. Sparse/short groups are retained with explicit fallback and undefined within-group R² where variance is zero.

## What this changes / next steps
Present the three-letter lookup as the stronger predictive baseline and qualify the chirality model's mechanistic interpretation. The coarse one-letter description alone understates the chirality model's residual information. A future independent test should hold out rule families and recalibrate across initial seeds. No claim that c alone diagnoses cube-root growth follows from these comparisons.